# 04 — Correlações e Modelagem

**Objetivo:** quantificar as relações entre o USD/BRL e as variáveis macroeconômicas por meio de análise de correlação e modelos de regressão.

**Entradas:** `data/processed/dataset_analitico.csv`  
**Saídas:** gráficos em `reports/figures/`, métricas dos modelos

---

## 0. Setup

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path('..') / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson
from scipy import stats
%matplotlib inline

from load import load_dataset
from indicators import (
    pearson_correlation,
    rolling_correlation,
    correlation_matrix,
)
from plots import (
    plot_correlation_heatmap,
    plot_rolling_correlation,
    set_style,
    PALETTE,
)

pd.set_option('display.float_format', '{:.4f}'.format)
set_style()

df = load_dataset(drop_na=False)
print(f'Dataset: {df.shape[0]} meses × {df.shape[1]} colunas')

## 1. Correlação de Pearson — visão geral

In [ ]:
corr = pearson_correlation(df, target='usd_brl')

print('Correlação de Pearson com USD/BRL (ordenado por |r|)')
print('=' * 45)
for col, val in corr.items():
    barra = '█' * int(abs(val) * 20)
    sinal = '+' if val >= 0 else '-'
    print(f'  {col:<28} {sinal}{abs(val):.3f}  {barra}')

In [ ]:
# gráfico de barras das correlações
fig, ax = plt.subplots(figsize=(9, 5))
colors = [PALETTE['accent'] if v > 0 else PALETTE['neutral'] for v in corr.values]
ax.barh(corr.index, corr.values, color=colors, edgecolor='white')
ax.axvline(0, color=PALETTE['muted'], linewidth=0.8)
ax.set_xlabel('Correlação de Pearson')
ax.set_title('Correlação com USD/BRL por Variável', fontweight='bold', pad=14)
ax.set_xlim(-1.1, 1.1)
ax.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('../reports/figures/04_correlacoes_pearson.png', dpi=300, bbox_inches='tight')
plt.show()

## 2. Heatmap de correlação completo

In [ ]:
fig = plot_correlation_heatmap(df)
plt.show()

## 3. Correlação rolling — estabilidade temporal

A correlação entre duas variáveis pode mudar ao longo do tempo. Aqui analisamos se a relação é estável ou varia por subperíodo.

In [ ]:
pares = [
    ('vix',          'VIX',               '04_rolling_corr_vix'),
    ('embi_brasil',  'EMBI+ Brasil',      '04_rolling_corr_embi'),
    ('dxy',          'DXY',               '04_rolling_corr_dxy'),
    ('spread_juros', 'Spread Juros',      '04_rolling_corr_spread'),
    ('ibovespa',     'Ibovespa',          '04_rolling_corr_ibovespa'),
]

for col, label, fname in pares:
    if col in df.columns:
        rc = rolling_correlation(df, 'usd_brl', col, window=12)
        fig = plot_rolling_correlation(
            rc,
            title=f'Correlação Rolling 12m — USD/BRL × {label}',
            filename=fname
        )
        plt.show()

## 4. Regressão linear simples

Para cada variável explicativa, estimamos:

$$\text{USD/BRL}_t = \beta_0 + \beta_1 \cdot X_t + \varepsilon_t$$

In [ ]:
regressores = [
    'selic', 'fed_funds', 'spread_juros', 'dxy',
    'vix', 'embi_brasil', 'ibovespa',
    'ipca_acumulado_12m', 'petroleo_wti'
]

resultados = []

for col in regressores:
    if col not in df.columns:
        continue
    sub = df[['usd_brl', col]].dropna()
    X = sm.add_constant(sub[col])
    y = sub['usd_brl']
    model = sm.OLS(y, X).fit()

    resultados.append({
        'variável':   col,
        'β₁':         model.params[col],
        'p-valor':    model.pvalues[col],
        'R²':         model.rsquared,
        'significativo': model.pvalues[col] < 0.05,
    })

df_ols = pd.DataFrame(resultados).sort_values('R²', ascending=False)
df_ols.round(4)

In [ ]:
# scatter plots para os 4 maiores R²
top4 = df_ols.head(4)['variável'].tolist()

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, col in enumerate(top4):
    sub = df[['usd_brl', col]].dropna()
    x, y = sub[col], sub['usd_brl']
    m, b, r, p, _ = stats.linregress(x, y)

    axes[i].scatter(x, y, alpha=0.4, s=18, color=PALETTE['neutral'])
    axes[i].plot(
        x, m * x + b,
        color=PALETTE['accent'], linewidth=1.5,
        label=f'R²={r**2:.3f}'
    )
    axes[i].set_xlabel(col, fontsize=9)
    axes[i].set_ylabel('USD/BRL', fontsize=9)
    axes[i].set_title(f'USD/BRL × {col}', fontweight='bold', fontsize=10)
    axes[i].legend(fontsize=8)
    axes[i].spines[['top', 'right']].set_visible(False)

plt.suptitle('Regressão Linear Simples — Top 4 R²', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('../reports/figures/04_scatter_regressoes.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Regressão linear múltipla

Modelo combinando as principais variáveis explicativas:

$$\text{USD/BRL}_t = \beta_0 + \beta_1 \cdot \text{DXY}_t + \beta_2 \cdot \text{EMBI}_t + \beta_3 \cdot \text{Spread}_t + \beta_4 \cdot \text{VIX}_t + \varepsilon_t$$

In [ ]:
features = ['dxy', 'embi_brasil', 'spread_juros', 'vix']
features_disponiveis = [f for f in features if f in df.columns]

df_model = df[['usd_brl'] + features_disponiveis].dropna()
X = sm.add_constant(df_model[features_disponiveis])
y = df_model['usd_brl']

modelo_multiplo = sm.OLS(y, X).fit()
print(modelo_multiplo.summary())

### 5.1 Verificação de multicolinearidade (VIF)

In [ ]:
vif_data = pd.DataFrame()
vif_data['variável'] = features_disponiveis
vif_data['VIF'] = [
    variance_inflation_factor(X.values, i + 1)
    for i in range(len(features_disponiveis))
]
vif_data['alerta'] = vif_data['VIF'].apply(
    lambda v: '⚠ multicolinearidade' if v > 10 else ('↗ moderado' if v > 5 else '✓ ok')
)
print('VIF — Variance Inflation Factor')
print('(VIF > 10 indica multicolinearidade severa)')
vif_data.round(2)

### 5.2 Diagnóstico dos resíduos

In [ ]:
residuos = modelo_multiplo.resid
fitted   = modelo_multiplo.fittedvalues
dw       = durbin_watson(residuos)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# resíduos vs fitted
axes[0].scatter(fitted, residuos, alpha=0.4, s=15, color=PALETTE['neutral'])
axes[0].axhline(0, color=PALETTE['accent'], linewidth=1, linestyle='--')
axes[0].set_xlabel('Valores ajustados')
axes[0].set_ylabel('Resíduos')
axes[0].set_title('Resíduos vs Ajustados', fontweight='bold')
axes[0].spines[['top', 'right']].set_visible(False)

# distribuição dos resíduos
axes[1].hist(residuos, bins=30, color=PALETTE['neutral'], edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Resíduo')
axes[1].set_title('Distribuição dos Resíduos', fontweight='bold')
axes[1].spines[['top', 'right']].set_visible(False)

# Q-Q plot
sm.qqplot(residuos, line='s', ax=axes[2], alpha=0.5, markersize=4)
axes[2].set_title('Q-Q Plot dos Resíduos', fontweight='bold')
axes[2].spines[['top', 'right']].set_visible(False)

plt.suptitle('Diagnóstico dos Resíduos — Regressão Múltipla', fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/04_diagnostico_residuos.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'Durbin-Watson: {dw:.3f}  (ideal: ~2.0 — sem autocorrelação)')
print(f'R² ajustado  : {modelo_multiplo.rsquared_adj:.4f}')
print(f'AIC          : {modelo_multiplo.aic:.2f}')

## 6. Conclusão

**Síntese dos resultados (preencher após execução):**

- Variável com maior correlação individual com USD/BRL: *[resultado]*
- R² do modelo múltiplo: *[resultado]*
- Fatores mais significativos (p < 0.05): *[resultado]*
- Durbin-Watson (autocorrelação): *[resultado]*
- Multicolinearidade detectada: *[resultado]*

> ⚠️ Os modelos são **exploratórios**. A relação câmbio–juros tem componente de endogeneidade que OLS não captura.

Próximo passo: **Notebook 05 — Conclusões e Insights**.